# ROM Rebuild + Inverse Characterisation

**Workflow:**
1. Discover every case in `data/01_training` — no hardcoded grids
2. POD basis from weighted snapshots (all valid pairs, all schedules)
3. Operator inference (time-weighted WLS + Tikhonov, adaptive regularisation)
4. RBF interpolation over $(\log_{10}k,\,S)$ space
5. Forward-error scan on new validation set
6. Inverse characterisation via Grid Search + Nelder-Mead + Two-Stage

**New validation grid** (all off training grid):
$k \in \{25,45,85,120,170,220,270\}$ mD,
$S \in \{0,1.5,2.5,4.5,5.5\}$,
schedules $\{1,2,3,5,9\}$


In [ ]:
import json, pickle, shutil, warnings, time, re
from pathlib import Path
import h5py, numpy as np, scipy.io as sio
import matplotlib.pyplot as plt
from matplotlib.ticker import NullLocator
from scipy.interpolate import RBFInterpolator
from scipy.optimize import minimize
from scipy.stats import norm as sp_norm
warnings.filterwarnings("ignore")
from matplotlib.lines import Line2D
import numpy as np

plt.rcParams.update({
    'text.usetex': True, 'font.family': 'serif',
    'font.serif': ['Computer Modern Roman'],
    'text.latex.preamble': r'\usepackage{amsmath} \usepackage{amssymb}',
    'font.size': 12, 'axes.labelsize': 12, 'axes.titlesize': 12,
    'legend.fontsize': 11, 'xtick.labelsize': 12, 'ytick.labelsize': 12,
    'xtick.direction': 'out', 'ytick.direction': 'out',
    'xtick.top': True, 'ytick.right': True,
    'xtick.minor.visible': False, 'ytick.minor.visible': False,
    'axes.linewidth': 1, 'lines.linewidth': 1.8, 'lines.markersize': 6,
    'axes.grid': False, 'figure.dpi': 150, 'savefig.dpi': 600,
    'savefig.bbox': 'tight', 'legend.frameon': True,
    'legend.framealpha': 1.0, 'legend.edgecolor': 'black',
    'legend.fancybox': False,
})
C1,C2,C3,C4   = "#1a5e9e","#0d6e56","#9b2a1a","#b07318"
C_MRST,C_ROM  = "#2C5F8A","#C0392B"
C_INV,C_GUESS = "#1a7a4a","#b07318"
C_TRUE,C_NM   = "#2C5F8A","#8B0000"

HERE         = Path(".").resolve()
TRAINING_DIR = HERE / "../data/01_training"
VAL_DIR      = HERE / "../data/02_validation"
ROM_DIR      = HERE / "../rom_vrate"
FIG_DIR      = HERE / "../figures_vrate"; FIG_DIR.mkdir(parents=True, exist_ok=True)
sf = lambda n: (plt.savefig(FIG_DIR/n, dpi=600, bbox_inches='tight'),
                print(f"  Saved: {FIG_DIR/n}"))

# Fixed physics (from run_single_case.m)
P_INIT = 4500.0; N_CELLS = 2500; N_STEPS = 350
T_START = 0.001; T_END = 5000.0
PHI=0.18; CT=1.2e-5; MU_CP=1.2; BO=1.15; H_FT=150.0
RW_FT=0.35; C_WBS=0.003; RE_FT=(50*30/2)*3.28084

# NEW validation grid (all off training grid)
K_VAL_NEW  = [25, 45, 85, 120, 170, 220, 270]
S_VAL_NEW  = [0, 1.5, 2.5, 4.5, 5.5]
SCHED_VAL  = [1, 2, 3, 5, 9]

# ── I/O helpers ──────────────────────────────────────────────────────
def load_mat(fp, key):
    try: return sio.loadmat(str(fp), variable_names=[key])[key]
    except NotImplementedError:
        with h5py.File(str(fp),"r") as f:
            a=f[key][()]; return a.T if a.ndim==2 else a

def load_well_data(d):
    wd=d/"well_data.mat"
    bhp=load_mat(wd,"bhp_psia").ravel(); t=load_mat(wd,"time_hr").ravel()
    q=load_mat(wd,"rate_STBd").ravel(); dp=load_mat(wd,"delta_p_psi").ravel()
    with open(d/"params.json") as f: pm=json.load(f)
    return bhp,t,q,dp,pm

def physics_ok(k,s):
    kh=k*H_FT; tw=170000*C_WBS*MU_CP*np.exp(0.14*s)/kh
    tp=(PHI*MU_CP*CT*RE_FT**2)/(0.000264*k)*0.1
    n_mtr=int(N_STEPS*np.log10(min(tp,T_END)/max(tw,T_START))/np.log10(T_END/T_START))
    return (tw<T_END) and (tw<tp) and (n_mtr>=15)

def bourdet(t,dp,L=0.2):
    lnt=np.log(t); d=np.full_like(dp,np.nan)
    for i in range(1,len(t)-1):
        il,ir=i-1,i+1
        while il>0 and lnt[i]-lnt[il]<L: il-=1
        while ir<len(t)-1 and lnt[ir]-lnt[i]<L: ir+=1
        dl,dr=lnt[i]-lnt[il],lnt[ir]-lnt[i]
        if dl>0 and dr>0:
            d[i]=((dp[i]-dp[il])/dl*dr+(dp[ir]-dp[i])/dr*dl)/(dl+dr)
    m=np.isfinite(d)&(d>0)&(dp>0); return t[m],dp[m],d[m]

print("Environment ready.")
print(f"Validation: k={K_VAL_NEW}, S={S_VAL_NEW}, sched={SCHED_VAL}")


In [ ]:
# ── Discover EVERYTHING in training folder — no hardcoded grids ─────
_pat = re.compile(r'^k(\d+)_s(\d+)_sched(\d+)$')
_cases = []
for d in sorted(TRAINING_DIR.iterdir()):
    if not d.is_dir(): continue
    m = _pat.match(d.name)
    if m and (d/"well_data.mat").exists():
        k,s,sid = int(m.group(1)), int(m.group(2)), int(m.group(3))
        _cases.append((k,s,sid,d))

K_GRID   = sorted(set(c[0] for c in _cases))
S_GRID   = sorted(set(c[1] for c in _cases))
SCHEDS   = sorted(set(c[2] for c in _cases))

print(f"Training cases found : {len(_cases)}")
print(f"k values  : {K_GRID}")
print(f"S values  : {S_GRID}")
print(f"Schedules : {SCHEDS}")

# Physics screening — use only (k,S) pairs with visible WBS+MTR
VALID_PAIRS = sorted(set((k,s) for k,s,_,_ in _cases if physics_ok(k,s)))
print(f"\nValid (k,S) pairs after physics screen: {len(VALID_PAIRS)}")

# Well-cell index (centre of 50×50 grid, 0-indexed Python)
WC_IDX = (25-1)*50 + 25 - 1

# ROM output directories
(ROM_DIR/"basis").mkdir(parents=True, exist_ok=True)
(ROM_DIR/"operators").mkdir(parents=True, exist_ok=True)
(ROM_DIR/"interpolants").mkdir(parents=True, exist_ok=True)


In [ ]:
# ── POD basis from ALL valid (k,S) pairs and ALL schedules ──────────
snap_cols, n_loaded = [], 0
for k,s in VALID_PAIRS:
    for sid in SCHEDS:
        cd = TRAINING_DIR/f"k{k:03d}_s{s:02d}_sched{sid}"
        try:
            X  = load_mat(cd/"snapshots_psia.mat","X_psia")
            dX = X - P_INIT
            _,t,_,_,_ = load_well_data(cd)
            dt = np.diff(t,prepend=0.); dt[0]=t[0]
            w  = np.sqrt(dt); w /= (w.mean()+1e-12)
            snap_cols.append(dX*w); n_loaded+=1
        except FileNotFoundError: pass

Z = np.hstack(snap_cols)
print(f"Snapshot matrix  : {Z.shape}  ({Z.nbytes/1e9:.2f} GB)  from {n_loaded} cases")
print("Computing SVD ...")
t0=time.perf_counter()
U,sigma,_ = np.linalg.svd(Z, full_matrices=False)
print(f"SVD done in {time.perf_counter()-t0:.1f}s  σ₁={sigma[0]:.2f}  σ_last={sigma[-1]:.2e}")

cum_energy = np.cumsum(sigma**2)/np.sum(sigma**2)
# Select r: cumulative energy >= 1-1e-8 (matches parent notebook criterion)
N_MODES = int(np.searchsorted(cum_energy, 1-1e-8)) + 1
N_MODES = max(N_MODES, 10)   # at least 10
Vn = U[:,:N_MODES]
print(f"N_MODES = {N_MODES}  (energy captured: {cum_energy[N_MODES-1]*100:.6f}%)")
print(f"Basis Vn : {Vn.shape}")

np.save(ROM_DIR/"basis"/"Vn.npy",    Vn)
np.save(ROM_DIR/"basis"/"sigma.npy", sigma)
print(f"Basis saved → {ROM_DIR/'basis'}/")


In [ ]:
def infer_AB(Vn, X_list, U_list, T_list, lam_mult=1.0):
    n_r = Vn.shape[1]
    Xhat_rows,U_rows,R_rows,W_rows=[],[],[],[]
    for X,u,t in zip(X_list,U_list,T_list):
        dX=X-P_INIT; Xhat=(Vn.T@dX).T
        dt=np.diff(t,prepend=0.); dt[0]=t[0]
        R=np.zeros_like(Xhat); R[0]=Xhat[0]/dt[0]
        for j in range(1,len(t)): R[j]=(Xhat[j]-Xhat[j-1])/dt[j]
        i_s=1; w=np.sqrt(dt[i_s:]); w/=(w.mean()+1e-12)
        Xhat_rows.append(Xhat[i_s:]); U_rows.append(u[i_s:].reshape(-1,1))
        R_rows.append(R[i_s:]); W_rows.append(w)
    Xhat_all=np.vstack(Xhat_rows); U_all=np.vstack(U_rows)
    R_all=np.vstack(R_rows); W_all=np.concatenate(W_rows)
    D=np.hstack([Xhat_all,U_all])
    col_norms=np.linalg.norm(D,axis=0)+1e-12
    D_w=(D/col_norms)*W_all[:,None]; R_w=R_all*W_all[:,None]
    lam=1e-6*lam_mult*np.trace(D_w.T@D_w)/D_w.shape[1]
    nc=D_w.shape[1]
    D_aug=np.vstack([D_w,np.sqrt(lam)*np.eye(nc)])
    R_aug=np.vstack([R_w,np.zeros((nc,n_r))])
    OT_sc,*_=np.linalg.lstsq(D_aug,R_aug,rcond=None)
    OT=OT_sc/col_norms[:,None]; O=OT.T
    return O[:,:n_r], O[:,n_r:n_r+1]

def infer_CD(Vn, X_list, U_list, BHP_list):
    C_hat=Vn[WC_IDX,:].reshape(1,-1); D_est=[]
    for X,u,bhp in zip(X_list,U_list,BHP_list):
        dX=X-P_INIT; Xhat=(Vn.T@dX).T; dBHP=(bhp-P_INIT).reshape(-1,1)
        grid_dp=(C_hat@Xhat.T).T; skin_dp=dBHP[1:]-grid_dp[1:]
        u_vec=u[1:].reshape(-1,1)
        D_est.append(np.vdot(u_vec,skin_dp)/(np.vdot(u_vec,u_vec)+1e-12))
    return C_hat, np.array([[np.mean(D_est)]])

ops_dir=ROM_DIR/"operators"
if ops_dir.exists(): shutil.rmtree(ops_dir)
ops_dir.mkdir(parents=True)
print(f"Inferring operators for {len(VALID_PAIRS)} (k,S) pairs ...\n")

results=[]
for k,s in VALID_PAIRS:
    tag=f"k{k:03d}_s{s:02d}"; out_d=ops_dir/tag; out_d.mkdir(exist_ok=True)
    X_list,U_list,T_list,BHP_list=[],[],[],[]
    for sid in SCHEDS:
        cd=TRAINING_DIR/f"k{k:03d}_s{s:02d}_sched{sid}"
        try:
            X=load_mat(cd/"snapshots_psia.mat","X_psia")
            bhp,t,q,_,pm=load_well_data(cd)
            X_list.append(X); U_list.append(q); T_list.append(t); BHP_list.append(bhp)
        except FileNotFoundError: pass
    if not X_list: print(f"  SKIP {tag}"); continue
    # Adaptive regularisation
    stable,lam_mult=False,1.0
    for _ in range(6):
        A,B=infer_AB(Vn,X_list,U_list,T_list,lam_mult)
        ev=np.linalg.eigvals(A)
        if np.all(ev.real<=1e-5): stable=True; break
        lam_mult*=5.0
    # Stability shift (matching parent notebook)
    emax=ev.real.max()
    if not stable: A=A-(emax+1e-6)*np.eye(N_MODES)
    C,D=infer_CD(Vn,X_list,U_list,BHP_list)
    for nm,arr in [("A_hat",A),("B_hat",B),("C_hat",C),("D_hat",D)]:
        np.save(out_d/f"{nm}.npy",arr)
    results.append(dict(tag=tag,k=k,S=s,stable=stable,eig_max=emax,n_scheds=len(X_list)))
    print(f"  {tag}  {'OK' if stable else 'SHIFT'}  λ_max={emax:.2e}  n_scheds={len(X_list)}")

print(f"\nDone: {len(results)} operators inferred")


In [ ]:
# ── RBF interpolation over (log10_k, S) ─────────────────────────────
pts_lst=[]; A_lst,B_lst,C_lst,D_lst=[],[],[],[]
for r in results:
    A_lst.append(np.load(ops_dir/r["tag"]/"A_hat.npy"))
    B_lst.append(np.load(ops_dir/r["tag"]/"B_hat.npy"))
    C_lst.append(np.load(ops_dir/r["tag"]/"C_hat.npy"))
    D_lst.append(np.load(ops_dir/r["tag"]/"D_hat.npy"))
    pts_lst.append([np.log10(r["k"]), r["S"]])

pts=np.array(pts_lst)
A_arr=np.array(A_lst); B_arr=np.array(B_lst)
C_arr=np.array(C_lst); D_arr=np.array(D_lst)

pts_min=pts.min(0); pts_rng=pts.max(0)-pts.min(0); pts_rng[pts_rng==0]=1.
pts_norm=(pts-pts_min)/pts_rng

def build_rbf(arr, pts_n, smoothing=0.01):
    flat=arr.reshape(len(arr),-1)
    ints=[RBFInterpolator(pts_n, flat[:,c], kernel="linear", smoothing=smoothing)
          for c in range(flat.shape[1])]
    return ints, arr.shape[1:]

def eval_op(interps, shape, k, s):
    pn=np.array([[(np.log10(k)-pts_min[0])/pts_rng[0],
                   (s-pts_min[1])/pts_rng[1]]])
    return np.array([f(pn)[0] for f in interps]).reshape(shape)

def _stab(A, m=1e-6):
    lr=np.max(np.real(np.linalg.eigvals(A)))
    return A-(lr+m)*np.eye(len(A)) if lr>=0 else A

print(f"Building RBF interpolants over {len(pts)} points ...")
t0=time.perf_counter()
A_interps,A_shape=build_rbf(A_arr,pts_norm)
B_interps,B_shape=build_rbf(B_arr,pts_norm)
C_interps,C_shape=build_rbf(C_arr,pts_norm)
D_interps,D_shape=build_rbf(D_arr,pts_norm)
print(f"Done in {time.perf_counter()-t0:.1f}s")

idir=ROM_DIR/"interpolants"; idir.mkdir(exist_ok=True)
for nm,obj in [("A",{"i":A_interps,"s":A_shape}),("B",{"i":B_interps,"s":B_shape}),
               ("C",{"i":C_interps,"s":C_shape}),("D",{"i":D_interps,"s":D_shape})]:
    with open(idir/f"{nm}_interp.pkl","wb") as f: pickle.dump(obj,f)
print(f"Interpolants saved → {idir}/")

def rom_bhp(k, s, t, q):
    A=_stab(eval_op(A_interps,A_shape,k,s))
    B=eval_op(B_interps,B_shape,k,s); C=eval_op(C_interps,C_shape,k,s)
    D=eval_op(D_interps,D_shape,k,s)
    dt=np.diff(t,prepend=0.); dt[0]=t[0]; In=np.eye(N_MODES)
    xk=np.zeros(N_MODES); Y=np.zeros(len(t))
    Cv=C.ravel(); Dv=float(D.ravel()[0])
    for j in range(len(t)):
        xk=np.linalg.solve(In-dt[j]*A, xk+dt[j]*B.ravel()*q[j])
        Y[j]=float(Cv@xk)+Dv*q[j]
    return P_INIT+Y

print("rom_bhp() ready.")


In [ ]:
# ── Forward error scan: new validation set ───────────────────────────
# Exclude sched=6 (shut-in — ROM cannot handle q→0 transitions reliably)
print("Forward error scan on new validation set ...\n")
print(f"{'Case':<32} {'k':>5} {'S':>5} {'sc':>3} {'fwd_err':>9}")
print("-"*58)

fwd_results=[]
for k in K_VAL_NEW:
    for s in S_VAL_NEW:
        for sched in SCHED_VAL:
            tag=f"k{k:03d}_s{s:02d}_sched{sched}"
            cd=VAL_DIR/tag
            if not (cd/"well_data.mat").exists(): continue
            try:
                bhp_fom,t,q,dp,pm=load_well_data(cd)
                bhp_rom=rom_bhp(k,s,t,q)
                err=(np.linalg.norm(bhp_rom-bhp_fom)/
                     (np.linalg.norm(P_INIT-bhp_fom)+1e-9))
                print(f"  {tag:<30} {k:>5} {s:>5} {sched:>3} {err:>9.4f}")
                fwd_results.append(dict(k=k,s=s,sched=sched,tag=tag,
                                        fwd_err=err,bhp=bhp_fom,t=t,q=q,pm=pm))
            except Exception as e:
                print(f"  {tag:<30}  ERROR: {e}")

fwd_results.sort(key=lambda x:x['fwd_err'])
print(f"\nTotal: {len(fwd_results)} cases evaluated")
print(f"\nBest 10:")
for r in fwd_results[:10]:
    print(f"  {r['tag']:<30} {r['fwd_err']:.4f}")


In [ ]:
# ── Fig: Forward error heatmap ───────────────────────────────────────

_ks=K_VAL_NEW; _ss=S_VAL_NEW; _sc=SCHED_VAL
mat=np.full((len(_ks)*len(_ss),len(_sc)),np.nan)
row_labels=[]
for ri,(k,s) in enumerate([(k,s) for k in _ks for s in _ss]):
    row_labels.append(f"k={k}, S={s}")
    for ci,sc in enumerate(_sc):
        match=[r for r in fwd_results if r['k']==k and r['s']==s and r['sched']==sc]
        if match: mat[ri,ci]=match[0]['fwd_err']

fig,ax=plt.subplots(figsize=(9,len(row_labels)*0.4+1.5))
cmap=plt.cm.RdYlGn_r.copy(); cmap.set_bad('lightgray')
im=ax.imshow(mat,aspect='auto',cmap=cmap,vmin=0,vmax=0.15,origin='upper')
ax.set_xticks(range(len(_sc))); ax.set_xticklabels([f"S{s}" for s in _sc])
ax.set_yticks(range(len(row_labels))); ax.set_yticklabels(row_labels,fontsize=9)
ax.set_xlabel("Schedule"); ax.set_ylabel("$(k,S)$ case")
ax.set_title(r"ROM forward error $\varepsilon_{\rm fwd}$ — new validation set")
for i in range(mat.shape[0]):
    for j in range(mat.shape[1]):
        if np.isfinite(mat[i,j]):
            col='w' if mat[i,j]>0.08 else 'k'
            ax.text(j,i,f"{mat[i,j]:.2f}",ha='center',va='center',fontsize=8,color=col)
plt.colorbar(im,ax=ax,label=r'$\varepsilon_{\rm fwd}$',shrink=0.6)
plt.tight_layout(); sf("fig_val01_fwd_error_heatmap.pdf"); plt.show()


In [ ]:
# ── Select inversion test case ───────────────────────────────────────
# Prefer: (a) low fwd_err, (b) sched=1 (constant rate — cleanest PTA)
K_train_set=set(k for k,s in VALID_PAIRS); S_train_set=set(s for k,s in VALID_PAIRS)
pref=[r for r in fwd_results if r['sched']==1 and r['fwd_err']<0.15]
if not pref: pref=fwd_results
case=pref[0]

bhp_obs=case['bhp']; t_obs=case['t']; q_obs=case['q']
K_TRUE=case['k'];    S_TRUE=case['s']
q_ref=float(np.atleast_1d(case['pm']['rates_STBd'])[0])
print(f"Inversion test case: {case['tag']}")
print(f"  k={K_TRUE} mD, S={S_TRUE}, sched={case['sched']}, fwd_err={case['fwd_err']:.4f}")

# Weights (log-time)
dt_w=np.diff(t_obs,prepend=0.); dt_w[0]=t_obs[0]
w=np.sqrt(dt_w); w/=w.mean()
def misfit(k,s):
    try: r=w*(rom_bhp(k,s,t_obs,q_obs)-bhp_obs); return 0.5*np.dot(r,r)
    except: return np.nan

# ══ METHOD A: GRID SEARCH ═════════════════════════════════════════════
NK,NS=60,50
k_vec=np.logspace(np.log10(5),np.log10(500),NK)
s_vec=np.linspace(0,15,NS)
kk,ss=np.meshgrid(k_vec,s_vec); JJ=np.full_like(kk,np.nan)
print(f"\nGrid search {NK}×{NS}={NK*NS} pts ...")
t0=time.perf_counter()
for i,si in enumerate(s_vec):
    for j,kj in enumerate(k_vec):
        try: JJ[i,j]=misfit(kj,si)
        except: pass
print(f"Done {time.perf_counter()-t0:.1f}s")

idx=np.unravel_index(np.nanargmin(JJ),JJ.shape)
K_GS=k_vec[idx[1]]; S_GS=s_vec[idx[0]]; J_min=JJ[idx]
sigma2=J_min/max(len(t_obs)-2,1)
if sigma2>1e-10:
    J_1sig=J_min+0.5*2.30*sigma2   # chi2(2) 68% joint CI
    in_cr=JJ<=J_1sig
    k_ci=(kk[in_cr].min(),kk[in_cr].max()); s_ci=(ss[in_cr].min(),ss[in_cr].max())
    ci_str=f"k∈[{k_ci[0]:.1f},{k_ci[1]:.1f}], S∈[{s_ci[0]:.2f},{s_ci[1]:.2f}]"
else:
    J_1sig=J_min+1e-6; k_ci=(K_GS,K_GS); s_ci=(S_GS,S_GS)
    ci_str="degenerate (J≈0)"

# ══ METHOD C: NELDER-MEAD ══════════════════════════════════════════════
_nc=[0]
def _mj(xi): _nc[0]+=1; return misfit(10**xi[0],xi[1])
nm=minimize(_mj,[np.log10(K_GS),S_GS],method='Nelder-Mead',
            options=dict(xatol=1e-5,fatol=1e-10,maxiter=1000,adaptive=True))
K_NM=10**nm.x[0]; S_NM=nm.x[1]
print(f"\nNelder-Mead: k={K_NM:.3f} mD, S={S_NM:.4f}  ({_nc[0]} calls, J*={nm.fun:.3e})")

# ══ METHOD B: TWO-STAGE (iterated) ════════════════════════════════════
S_NOM=float(np.clip(np.mean(list(S_train_set)),0,15))  # mean training S as prior
J_k=np.array([misfit(kj,S_NOM) for kj in k_vec]); K_B1=k_vec[np.nanargmin(J_k)]
J_s=np.array([misfit(K_B1,si)  for si in s_vec]); S_B1=s_vec[np.nanargmin(J_s)]
J_k2=np.array([misfit(kj,S_B1) for kj in k_vec]); K_B=k_vec[np.nanargmin(J_k2)]
J_s2=np.array([misfit(K_B,si)  for si in s_vec]); S_B=s_vec[np.nanargmin(J_s2)]

# Horner: Bourdet flat level in MTR window
dp_c=np.clip(P_INIT-bhp_obs,1e-2,None)
tb,_,db=bourdet(t_obs,dp_c,L=0.15)
t_wbs=170000*C_WBS*MU_CP*np.exp(0.14*S_NM)/(K_NM*H_FT)
t_pss=(PHI*MU_CP*CT*RE_FT**2)/(0.000264*K_NM)*0.1
mtr=(tb>t_wbs)&(tb<t_pss*0.5)
dp_flat=float(np.nanmedian(db[mtr])) if mtr.any() else float(np.nanmedian(db))
k_horner=70.6*q_ref*MU_CP*BO/(dp_flat*H_FT)

print(f"\n{'Method':<24} {'k [mD]':>8} {'k%err':>7} {'S':>7} {'S err':>7} {'calls':>7}")
print("-"*60)
print(f"{'True (FOM)':<24} {K_TRUE:>8.1f} {'—':>7} {S_TRUE:>7.2f} {'—':>7} {'—':>7}")
print(f"{'A: Grid Search':<24} {K_GS:>8.2f} {abs(K_GS-K_TRUE)/K_TRUE*100:>6.1f}%"
      f" {S_GS:>7.2f} {abs(S_GS-S_TRUE):>7.3f} {NK*NS:>7}")
print(f"{'C: Nelder-Mead':<24} {K_NM:>8.3f} {abs(K_NM-K_TRUE)/K_TRUE*100:>6.2f}%"
      f" {S_NM:>7.4f} {abs(S_NM-S_TRUE):>7.4f} {_nc[0]:>7}")
print(f"{'B: Two-Stage':<24} {K_B:>8.2f}  {abs(K_B-K_TRUE)/K_TRUE*100:>6.1f}%"
      f" {S_B:>7.2f} {abs(S_B-S_TRUE):>7.3f} {'~120':>7}")
print(f"{'Horner (MTR)':<24} {k_horner:>8.2f} {abs(k_horner-K_TRUE)/K_TRUE*100:>6.1f}%")
print(f"\n68% CI: {ci_str}")
print(f"ROM fwd_err: {case['fwd_err']:.4f}")


In [ ]:
fig,ax=plt.subplots(figsize=(7,5.5))
Jlog=np.log10(np.clip(JJ,1e-4,None))
vmin,vmax=np.nanpercentile(Jlog,2),np.nanpercentile(Jlog,98)
cf=ax.contourf(kk,ss,Jlog,levels=30,cmap='RdYlGn_r',vmin=vmin,vmax=vmax,alpha=0.88)
ax.contour(kk,ss,Jlog,levels=12,colors='k',linewidths=0.4,alpha=0.3)
plt.colorbar(cf,ax=ax,label=r'$\log_{10}\,\mathcal{J}(k,\,S)$')
if sigma2>1e-10:
    ax.contour(kk,ss,JJ,levels=[J_1sig],colors=[C_INV],linewidths=1.8,linestyles='--')

ax.plot(K_TRUE,S_TRUE,'*',color=C_TRUE,ms=14,mec='k',mew=0.8,zorder=12,
        label=f'FOM ($k={K_TRUE}$, $S={S_TRUE}$)')
ax.plot(K_GS,S_GS,'D',color=C_INV,ms=9,mec='k',mew=0.7,zorder=10,
        label=f'Grid search ($k={K_GS:.1f}$, $S={S_GS:.2f}$)')
ax.plot(K_NM,S_NM,'P',color=C_NM,ms=10,mec='k',mew=0.8,zorder=11,
        label=f'Nelder-Mead ($k={K_NM:.1f}$, $S={S_NM:.2f}$)')
ax.plot(K_B,S_B,'s',color=C_GUESS,ms=8,mec='k',mew=0.7,zorder=9,
        label=f'Two-stage ($k={K_B:.1f}$, $S={S_B:.2f}$)')


h,l=ax.get_legend_handles_labels()
if sigma2>1e-10:
    h.append(Line2D([0],[0],color=C_INV,ls='--',lw=1.8)); l.append(r'68\% CI')
ax.legend(h,l,fontsize=9,loc='upper left',framealpha=0.95)
ax.set_xscale('log'); ax.set_xlim(5,500); ax.set_ylim(0,15)
ax.set_xlabel(r'$k$ [mD]'); ax.set_ylabel(r'$S$ [-]')
ax.xaxis.set_minor_locator(NullLocator()); ax.yaxis.set_minor_locator(NullLocator())
ax.set_title(f"Misfit — {case['tag']}  (fwd\\_err={case['fwd_err']:.3f})",fontsize=10)
plt.tight_layout(); sf("fig_inv01_landscape.pdf"); plt.show()


In [ ]:
fig,axes=plt.subplots(1,2,figsize=(10,4))
axes[0].semilogy(k_vec,J_k,color=C1,lw=2.2)
axes[0].axvline(K_B,color=C_NM,ls='--',lw=1.8,label=f'Stage-1 $k={K_B:.1f}$')
axes[0].axvline(K_TRUE,color=C_TRUE,ls=':',lw=1.8,label=f'True $k={K_TRUE}$')
axes[0].set_xscale('log'); axes[0].set_xlabel(r'$k$ [mD]')
axes[0].set_ylabel(r'$\mathcal{J}(k,S_{\rm nom})$')
axes[0].set_title(f'(a) $k$ search at $S_{{\\rm nom}}={S_NOM:.1f}$',loc='left')
axes[0].legend(fontsize=10)
axes[0].xaxis.set_minor_locator(NullLocator()); axes[0].yaxis.set_minor_locator(NullLocator())

axes[1].semilogy(s_vec,J_s2,color=C1,lw=2.2)
axes[1].axvline(S_B,color=C_NM,ls='--',lw=1.8,label=f'Stage-2 $S={S_B:.2f}$')
axes[1].axvline(S_TRUE,color=C_TRUE,ls=':',lw=1.8,label=f'True $S={S_TRUE}$')
axes[1].set_xlabel(r'$S$ [-]'); axes[1].set_ylabel(r'$\mathcal{J}(\hat{k},S)$')
axes[1].set_title(f'(b) $S$ search at $\\hat{{k}}={K_B:.1f}$ mD',loc='left')
axes[1].legend(fontsize=10)
axes[1].xaxis.set_minor_locator(NullLocator()); axes[1].yaxis.set_minor_locator(NullLocator())
plt.tight_layout(); sf("fig_inv02_slices.pdf"); plt.show()


In [ ]:
bhp_nm=rom_bhp(K_NM,S_NM,t_obs,q_obs)
bhp_gs=rom_bhp(K_GS,S_GS,t_obs,q_obs)
bhp_b =rom_bhp(K_B, S_B, t_obs,q_obs)
dp_obs_c=np.clip(P_INIT-bhp_obs,1e-2,None)
dp_nm_c =np.clip(P_INIT-bhp_nm, 1e-2,None)
dp_gs_c =np.clip(P_INIT-bhp_gs, 1e-2,None)
dp_b_c  =np.clip(P_INIT-bhp_b,  1e-2,None)

fig,axes=plt.subplots(1,3,figsize=(13,4))
series=[(bhp_obs,dp_obs_c,C_TRUE, '-', f'FOM ($k={K_TRUE}$,$S={S_TRUE}$)'),
        (bhp_nm, dp_nm_c, C_NM,  '-.', f'NM ($k={K_NM:.1f}$,$S={S_NM:.2f}$)'),
        (bhp_gs, dp_gs_c, C_INV, '--', f'GS ($k={K_GS:.1f}$,$S={S_GS:.2f}$)'),
        (bhp_b,  dp_b_c, C_GUESS,':', f'2S ($k={K_B:.1f}$,$S={S_B:.2f}$)')]
for bhp_,dp_,col_,ls_,lbl_ in series:
    axes[0].semilogx(t_obs,bhp_,color=col_,lw=2.2,ls=ls_,label=lbl_)
    axes[1].loglog(t_obs,dp_, color=col_,lw=2.2,ls=ls_,label=lbl_)
    tb_,_,db_=bourdet(t_obs,dp_)
    if len(db_)>3: axes[2].loglog(tb_,db_,color=col_,lw=2.2,ls=ls_,label=lbl_)
for k_,col_ in [(K_TRUE,C_TRUE),(K_NM,C_NM),(K_GS,C_INV),(K_B,C_GUESS)]:
    axes[2].axhline(70.6*q_ref*MU_CP*BO/(k_*H_FT),color=col_,ls=':',lw=1.1,alpha=0.6)
ylbls=[r'$p_{wf}$ [psia]',r'$\Delta p_{wf}$ [psi]',r"$t\partial\Delta p/\partial t$ [psi]"]
for i,(ax,yl) in enumerate(zip(axes,ylbls)):
    ax.set_xlabel(r'$t$ [hr]'); ax.set_ylabel(yl)
    ax.set_title(f'({chr(97+i)})',loc='left'); ax.set_xlim(t_obs[0],t_obs[-1])
    ax.xaxis.set_minor_locator(NullLocator()); ax.yaxis.set_minor_locator(NullLocator())
    ax.legend(fontsize=8)
plt.tight_layout(); sf("fig_inv03_bhp_match.pdf"); plt.show()


In [ ]:
res_nm=bhp_nm-bhp_obs; res_gs=bhp_gs-bhp_obs; res_b=bhp_b-bhp_obs
rmse={n:np.sqrt(np.mean(r**2)) for n,r in [('NM',res_nm),('GS',res_gs),('B',res_b)]}

fig,axes=plt.subplots(1,2,figsize=(10,3.8))
axes[0].semilogx(t_obs,res_nm,color=C_NM,   lw=2.2,     label=f'NM  RMSE={rmse["NM"]:.2f} psi')
axes[0].semilogx(t_obs,res_gs,color=C_INV,  lw=1.8,ls='--',label=f'GS  RMSE={rmse["GS"]:.2f} psi')
axes[0].semilogx(t_obs,res_b, color=C_GUESS,lw=1.8,ls=':',label=f'2S  RMSE={rmse["B"]:.2f} psi')
axes[0].axhline(0,color='k',lw=0.8,ls='--')
for y in [2,-2]: axes[0].axhline(y,color='gray',lw=0.8,ls='--',alpha=0.5)
axes[0].set_xlabel(r'$t$ [hr]'); axes[0].set_ylabel(r'$\hat{p}_{wf}-p^{\rm obs}_{wf}$ [psi]')
axes[0].set_xlim(t_obs[0],t_obs[-1]); axes[0].set_title('(a)',loc='left')
axes[0].legend(fontsize=9); axes[0].xaxis.set_minor_locator(NullLocator())
axes[0].yaxis.set_minor_locator(NullLocator())

rn=res_nm/(np.std(res_nm)+1e-9); bins=np.linspace(-4,4,22)
axes[1].hist(rn,bins=bins,color=C_NM,edgecolor='k',lw=0.4,alpha=0.8,density=True)
axes[1].plot(np.linspace(-4,4,200),sp_norm.pdf(np.linspace(-4,4,200)),'k--',lw=1.5,
             label=r'$\mathcal{N}(0,1)$')
axes[1].set_xlabel(r'$(p_{\rm rec}-p_{\rm obs})/\hat\sigma$')
axes[1].set_ylabel('Density'); axes[1].set_title('(b)',loc='left')
axes[1].legend(fontsize=10); axes[1].xaxis.set_minor_locator(NullLocator())
axes[1].yaxis.set_minor_locator(NullLocator())
plt.tight_layout(); sf("fig_inv04_residuals.pdf"); plt.show()

# Summary table
fig,ax=plt.subplots(figsize=(13,2.4)); ax.axis('off')
cols=['Parameter','True (FOM)','A: Grid Search','C: Nelder-Mead',
      'B: Two-Stage','Horner','A Err','C Err','B Err']
rows=[
    [r'$k$ [mD]',f'{K_TRUE}',f'{K_GS:.1f}',f'{K_NM:.2f}',f'{K_B:.1f}',f'{k_horner:.1f}',
     f'{abs(K_GS-K_TRUE)/K_TRUE*100:.1f}\\%',
     f'{abs(K_NM-K_TRUE)/K_TRUE*100:.2f}\\%',
     f'{abs(K_B-K_TRUE)/K_TRUE*100:.1f}\\%'],
    [r'$S$ [-]',f'{S_TRUE}',f'{S_GS:.2f}',f'{S_NM:.3f}',f'{S_B:.2f}','—',
     f'{abs(S_GS-S_TRUE):.3f}',f'{abs(S_NM-S_TRUE):.4f}',f'{abs(S_B-S_TRUE):.3f}'],
]
tbl=ax.table(cellText=rows,colLabels=cols,cellLoc='center',loc='center',bbox=[0,0,1,1])
tbl.auto_set_font_size(False); tbl.set_fontsize(10)
for (r_,c_),cell in tbl.get_celld().items():
    cell.set_edgecolor('#CCCCCC'); cell.set_linewidth(0.5)
    if r_==0: cell.set_facecolor('#2C5F8A'); cell.set_text_props(color='white',fontweight='bold')
    elif r_%2==0: cell.set_facecolor('#EBF2FA')
plt.tight_layout(); sf("fig_inv05_table.pdf"); plt.show()


### Notes

**ROM rebuild.** The training grid ($k$, $S$, schedules) is discovered entirely at runtime
from `data/01_training` — no values are hardcoded. Every valid $(k,S)$ pair (passing the
physics screen: visible WBS + $\geq15$ MTR steps) contributes to the POD basis and operator
inference. The mode count $r$ is selected by the $1-10^{-8}$ energy threshold.

**New validation grid.** All 7 $k$ values (25, 45, 85, 120, 170, 220, 270 mD) are off the
training grid. $S \in \{0, 1.5, 2.5, 4.5, 5.5\}$ mixes seen and unseen skin values.
Schedules 1,2,3,5 were used in training; schedule 9 (4-period, non-monotonic) was not.

**Inversion.** The ROM acts as a surrogate forward model inside the optimisation loop.
Grid search (60×50) maps the full misfit landscape in ~15\,s vs >3\,hours with FOM.
Nelder-Mead refines the grid minimum to near-exact precision in ~100 ROM calls (<1\,s).
Recovery accuracy is bounded by the ROM forward error $\varepsilon_{\rm fwd}$ at the
true parameter point — cases with $\varepsilon_{\rm fwd}<5\%$ give reliable inversion.
